# Fase 1 — Uji Statistik Deret Sensor Kebun Kopi

**Tujuan.** Menetapkan sifat statistik deret sebelum satu model pun dilatih, agar
pemilihan model pada Fase 2 punya dasar, bukan coba-coba.

Notebook ini menjawab tiga pertanyaan:

1. Apakah deret **stasioner**, dan perlukah pembedaan? → ADF & KPSS
2. Adakah **struktur yang dapat dipelajari**, atau hanya derau? → ACF/PACF & Ljung-Box
3. Apakah deret berperilaku seperti **jalan acak**? → Variance Ratio (Lo & MacKinlay)

> **Gerbang keputusan.** Bila deret ternyata jalan acak murni di semua resolusi,
> model kompleks tidak punya peluang mengalahkan Naive dan paper harus dibingkai
> ulang sebagai temuan negatif. Lebih baik diketahui sekarang daripada setelah
> melatih puluhan model.

**Data.** Dataset beku `sesi_20260824-20260829.csv` — satu sesi kontinu 134,24 jam
(7.992 pembacaan) dari channel ThingSpeak 1848413. Notebook memverifikasi SHA-256
berkas sebelum memakainya, sehingga hasil yang Anda peroleh dapat dipastikan
berasal dari data yang sama persis.

Jalankan: **Runtime → Run all** (±2–3 menit).

In [ ]:
# Pustaka. Colab sudah menyediakan semuanya; baris ini hanya jaring pengaman.
!pip install -q statsmodels pandas numpy scipy matplotlib
print('siap')

## 1. Muat dataset beku dan verifikasi keasliannya

Dataset diambil dari repositori publik, lalu sidik jari SHA-256-nya dicocokkan
dengan nilai yang tercatat di `MANIFEST.json`. Bila tidak cocok, berarti berkas
berubah dan hasil analisis tidak boleh dibandingkan dengan laporan sebelumnya.

In [ ]:
import io, os, math, json, hashlib, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from scipy import stats as sstats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings('ignore')

URL = 'https://raw.githubusercontent.com/jejenfisika-stack/IoT-Kopi_Web3/main/dataset/sesi_20260824-20260829.csv'
SHA_HARAP = '92025679db8c2ab50123ebe812ace209f60643cd0c229ac16aca5e1c44454690'

OUT = 'hasil_fase1'
os.makedirs(OUT, exist_ok=True)

VAR = {'suhu': ('Suhu udara', '\u00b0C'),
       'udara': ('Kelembaban udara', '%'),
       'tanah': ('Kelembaban tanah', '%')}
RESOLUSI = ['5min', '10min', '30min', '1h']
RULE_UTAMA = '5min'      # resolusi utama untuk ACF/PACF & Ljung-Box
ALPHA = 0.05

mentah = requests.get(URL, timeout=180).content
sha = hashlib.sha256(mentah).hexdigest()
print('SHA-256 berkas :', sha)
print('cocok manifes  :', sha == SHA_HARAP)
if sha != SHA_HARAP:
    print('\n\u26a0 PERINGATAN: berkas berbeda dari yang tercatat di manifes.')

df = pd.read_csv(io.BytesIO(mentah), parse_dates=['waktu']).set_index('waktu').sort_index()
print(f'\nbacaan : {len(df)}')
print(f'rentang: {df.index.min()}  ->  {df.index.max()}')
print(f'durasi : {(df.index.max()-df.index.min()).total_seconds()/3600:.2f} jam')
df.head()

## 2. Statistik deskriptif

Perhatikan **skewness** dan **kurtosis**: nilai yang ekstrem menandakan ada
pencilan berat yang akan mengganggu pelatihan model bila tidak ditangani di
Fase 2.

In [ ]:
rows = []
for k, (nama, sat) in VAR.items():
    s = df[k].dropna()
    rows.append({'Variabel': nama, 'Satuan': sat, 'N': len(s),
                 'Mean': round(s.mean(), 2), 'SD': round(s.std(ddof=1), 2),
                 'Min': round(s.min(), 2), 'Maks': round(s.max(), 2),
                 'Skewness': round(s.skew(), 3), 'Kurtosis': round(s.kurtosis(), 3)})
tab_desk = pd.DataFrame(rows)
print('TABEL 1 — Statistik deskriptif (data mentah)')
display(tab_desk)

## 3. Penyeragaman jarak antar pengamatan

Uji stasioneritas, autokorelasi, dan rasio ragam **mensyaratkan jarak antar
pengamatan yang seragam**. Data mentah berjarak ±1 menit namun tidak persis,
sehingga di-*resample* ke beberapa resolusi.

Bin yang kosong diisi interpolasi **maksimal dua bin berturut-turut**; celah yang
lebih panjang dibiarkan hilang agar tidak menciptakan pola semu.

In [ ]:
def seragam(kolom, rule, batas_isi=2):
    """Resample ke jarak seragam; isi celah pendek saja."""
    s = df[kolom].resample(rule).mean()
    n_bin, n_kosong = len(s), int(s.isna().sum())
    s = s.interpolate(limit=batas_isi, limit_direction='both').dropna()
    return s, n_bin, n_kosong

rows = []
for rule in RESOLUSI:
    for k, (nama, _) in VAR.items():
        s, n_bin, n_kosong = seragam(k, rule)
        rows.append({'Resolusi': rule, 'Variabel': nama, 'Bin': n_bin,
                     'Kosong': n_kosong, 'Terpakai': len(s),
                     'Kelengkapan (%)': round(100*(n_bin-n_kosong)/n_bin, 2)})
tab_leng = pd.DataFrame(rows)
print('TABEL 2 — Kelengkapan setelah penyeragaman jarak')
display(tab_leng)

## 4. Stasioneritas — ADF dan KPSS

Kedua uji ini punya **hipotesis nol yang berlawanan**, sehingga saling melengkapi:

| Uji | H₀ | Menolak H₀ berarti |
|---|---|---|
| ADF | ada akar unit (tidak stasioner) | deret **stasioner** |
| KPSS | deret stasioner | deret **tidak stasioner** |

Cara membaca gabungannya:

| ADF | KPSS | Kesimpulan |
|---|---|---|
| tolak | tidak tolak | **stasioner** |
| tidak tolak | tolak | **akar unit** — perlu pembedaan |
| tolak | tolak | tren-stasioner atau heteroskedastik |
| tidak tolak | tidak tolak | bukti tidak memadai |

Diuji pada **aras** (nilai asli) dan **beda-1** (selisih antar pengamatan).

In [ ]:
def uji_adf_kpss(x):
    out = {}
    try:
        a = adfuller(x, autolag='AIC')
        out['adf_stat'], out['adf_p'] = a[0], a[1]
    except Exception:
        out['adf_stat'] = out['adf_p'] = np.nan
    try:
        k = kpss(x, regression='c', nlags='auto')
        out['kpss_stat'], out['kpss_p'] = k[0], k[1]
    except Exception:
        out['kpss_stat'] = out['kpss_p'] = np.nan
    return out

def putusan(adf_p, kpss_p, a=ALPHA):
    if not np.isfinite(adf_p) or not np.isfinite(kpss_p): return 'tidak konklusif'
    tolak_ur, tolak_st = adf_p < a, kpss_p < a
    if tolak_ur and not tolak_st: return 'stasioner'
    if not tolak_ur and tolak_st: return 'akar unit'
    if tolak_ur and tolak_st:     return 'tren-stasioner / heteroskedastik'
    return 'bukti tidak memadai'

rows = []
for rule in RESOLUSI:
    for k, (nama, _) in VAR.items():
        s, _, _ = seragam(k, rule)
        for lab, x in (('aras', s.values), ('beda-1', np.diff(s.values))):
            if len(x) < 20 or np.std(x) < 1e-12: continue
            r = uji_adf_kpss(x)
            rows.append({'Resolusi': rule, 'Variabel': nama, 'Deret': lab, 'N': len(x),
                         'ADF': round(r['adf_stat'], 3), 'p(ADF)': round(r['adf_p'], 4),
                         'KPSS': round(r['kpss_stat'], 3), 'p(KPSS)': round(r['kpss_p'], 4),
                         'Putusan': putusan(r['adf_p'], r['kpss_p'])})
tab_stat = pd.DataFrame(rows)
print('TABEL 3 — Uji stasioneritas ADF & KPSS')
display(tab_stat)

## 5. Struktur autokorelasi — ACF, PACF, dan Ljung-Box

**ACF** memperlihatkan seberapa jauh pengaruh masa lalu bertahan; **PACF**
membantu menebak orde AR. Pada deret dengan siklus harian, ACF akan menunjukkan
gelombang berulang.

**Ljung-Box** menguji apakah autokorelasi secara keseluruhan berbeda dari nol.
Bila deret **beda-1** masih berkorelasi, berarti masih ada struktur yang dapat
dipelajari melampaui jalan acak — kabar baik bagi model.

In [ ]:
fig, ax = plt.subplots(len(VAR), 2, figsize=(12, 3.1*len(VAR)))
for i, (k, (nama, _)) in enumerate(VAR.items()):
    s, _, _ = seragam(k, RULE_UTAMA)
    nl = min(48, len(s)//2 - 1)
    plot_acf(s.values, lags=nl, ax=ax[i, 0], title=f'ACF — {nama} (aras, {RULE_UTAMA})')
    plot_pacf(s.values, lags=min(nl, len(s)//2 - 1), ax=ax[i, 1], method='ywm',
              title=f'PACF — {nama} (aras, {RULE_UTAMA})')
plt.tight_layout(); plt.savefig(f'{OUT}/acf_pacf_aras.png', dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(len(VAR), 2, figsize=(12, 3.1*len(VAR)))
for i, (k, (nama, _)) in enumerate(VAR.items()):
    s, _, _ = seragam(k, RULE_UTAMA)
    x = np.diff(s.values)
    nl = min(48, len(x)//2 - 1)
    plot_acf(x, lags=nl, ax=ax[i, 0], title=f'ACF — {nama} (beda-1, {RULE_UTAMA})')
    plot_pacf(x, lags=min(nl, len(x)//2 - 1), ax=ax[i, 1], method='ywm',
              title=f'PACF — {nama} (beda-1, {RULE_UTAMA})')
plt.tight_layout(); plt.savefig(f'{OUT}/acf_pacf_beda.png', dpi=150); plt.show()

In [ ]:
rows = []
for k, (nama, _) in VAR.items():
    s, _, _ = seragam(k, RULE_UTAMA)
    for lab, x in (('aras', s.values), ('beda-1', np.diff(s.values))):
        lags = [l for l in (6, 12, 24, 48) if l < len(x)//2]
        lb = acorr_ljungbox(x, lags=lags, return_df=True)
        for l in lags:
            rows.append({'Variabel': nama, 'Deret': lab, 'Lag': l,
                         'Q': round(float(lb.loc[l, 'lb_stat']), 2),
                         'p': round(float(lb.loc[l, 'lb_pvalue']), 4),
                         'Berkorelasi?': 'ya' if lb.loc[l, 'lb_pvalue'] < ALPHA else 'tidak (derau putih)'})
tab_lb = pd.DataFrame(rows)
print(f'TABEL 4 — Ljung-Box pada resolusi {RULE_UTAMA}')
display(tab_lb)

## 6. Uji Variance Ratio (Lo & MacKinlay, 1988)

Inilah uji formal terhadap hipotesis **jalan acak**. Gagasannya: pada jalan acak,
ragam selisih $q$-periode tepat $q$ kali ragam selisih satu periode, sehingga
rasionya $VR(q) = 1$.

| Nilai | Arti | Implikasi bagi peramalan |
|---|---|---|
| $VR \approx 1$ | jalan acak | nilai terakhir sudah penduga terbaik — Naive sulit dikalahkan |
| $VR > 1$ | persistensi / tren | ada momentum yang dapat dipelajari model |
| $VR < 1$ | pembalikan arah | ada efek kembali ke rata-rata |

Dipakai statistik $Z^*$ yang **tahan heteroskedastisitas**, sebab ragam data
sensor berubah sepanjang hari.

Sel berikut **memvalidasi implementasinya lebih dulu** lewat simulasi Monte Carlo:
pada jalan acak buatan, tingkat penolakan harus mendekati 5%; pada AR(1)
stasioner, kuasanya harus tinggi. Tanpa langkah ini, angka VR tidak layak
dipercaya.

In [ ]:
def variance_ratio(x, q):
    """VR(q) tumpang-tindih. Mengembalikan (VR, Z homoskedastik, Z* tahan heteroskedastisitas)."""
    x = np.asarray(x, float); n = len(x) - 1
    if n < q*2 or q < 2: return np.nan, np.nan, np.nan
    mu = (x[-1] - x[0]) / n
    d1 = np.diff(x) - mu
    sa = np.sum(d1**2) / (n - 1)
    if sa <= 0: return np.nan, np.nan, np.nan
    dq = x[q:] - x[:-q] - q*mu
    m = q * (n - q + 1) * (1 - q/n)
    if m <= 0: return np.nan, np.nan, np.nan
    vr = (np.sum(dq**2) / m) / sa
    phi = 2.0*(2*q - 1)*(q - 1) / (3.0*q*n)
    z = (vr - 1) / math.sqrt(phi) if phi > 0 else np.nan
    pembagi = np.sum(d1**2)**2
    phi_b = 0.0
    for j in range(1, q):
        num = np.sum((d1[j:]**2) * (d1[:-j]**2))
        phi_b += ((2.0*(q - j)/q)**2) * (num/pembagi if pembagi > 0 else 0.0)
    zb = (vr - 1) / math.sqrt(phi_b) if phi_b > 0 else np.nan
    return vr, z, zb

def mc_validasi(nsim=400, n=400, seed=1):
    g = np.random.default_rng(seed); tolak_rw = tolak_ar = 0
    for _ in range(nsim):
        rw = np.cumsum(g.normal(size=n))
        _, _, zb = variance_ratio(rw, 4)
        if np.isfinite(zb) and abs(zb) > 1.96: tolak_rw += 1
        e = g.normal(size=n); ar = np.zeros(n)
        for t in range(1, n): ar[t] = 0.3*ar[t-1] + e[t]
        _, _, zb2 = variance_ratio(ar, 4)
        if np.isfinite(zb2) and abs(zb2) > 1.96: tolak_ar += 1
    return tolak_rw/nsim, tolak_ar/nsim

taraf, kuasa = mc_validasi()
print(f'Taraf pada jalan acak buatan (harus \u2248 0,05) : {taraf:.3f}')
print(f'Kuasa pada AR(1) stasioner (harus tinggi)      : {kuasa:.3f}')
print('\nImplementasi layak dipakai.' if taraf < 0.10 and kuasa > 0.8 else '\n\u26a0 Periksa kembali implementasi.')

In [ ]:
rows = []
for rule in RESOLUSI:
    for k, (nama, _) in VAR.items():
        s, _, _ = seragam(k, rule)
        for q in (2, 4, 8, 16):
            vr, z, zb = variance_ratio(s.values, q)
            if not np.isfinite(vr): continue
            p = 2*(1 - sstats.norm.cdf(abs(zb))) if np.isfinite(zb) else np.nan
            rows.append({'Resolusi': rule, 'Variabel': nama, 'q': q,
                         'VR': round(vr, 3),
                         'Z*': round(zb, 3) if np.isfinite(zb) else np.nan,
                         'p': round(p, 4) if np.isfinite(p) else np.nan,
                         'Jalan acak?': ('ditolak' if np.isfinite(p) and p < ALPHA else 'tidak ditolak')})
tab_vr = pd.DataFrame(rows)
print('TABEL 5 — Uji Variance Ratio')
display(tab_vr)

## 7. Gerbang keputusan dan hipotesis untuk Fase 2

Sel ini merangkum bukti dan menerjemahkannya menjadi **hipotesis per variabel**
yang akan diuji pada Fase 2. Inilah yang membedakan paper berbasis bukti dari
sekadar laporan "kami mencoba beberapa model".

In [ ]:
ringkas = []
for k, (nama, _) in VAR.items():
    vr_k = tab_vr[tab_vr.Variabel == nama]
    ditolak = int((vr_k['Jalan acak?'] == 'ditolak').sum()); total = len(vr_k)
    vr_rerata = float(vr_k['VR'].mean())
    st_k = tab_stat[(tab_stat.Variabel == nama) & (tab_stat.Deret == 'aras')]
    akar_unit = int((st_k['Putusan'] == 'akar unit').sum())

    if ditolak / max(1, total) >= 0.5:
        dugaan = 'Model pembelajaran BERPELUANG mengungguli Naive'
    elif ditolak == 0:
        dugaan = 'Mendekati jalan acak — Naive diperkirakan sulit dikalahkan'
    else:
        dugaan = 'Campuran — keunggulan model bergantung resolusi'
    ringkas.append({'Variabel': nama,
                    'Jalan acak ditolak': f'{ditolak}/{total}',
                    'VR rerata': round(vr_rerata, 2),
                    'Resolusi berakar unit': f'{akar_unit}/{len(st_k)}',
                    'Hipotesis Fase 2': dugaan})
tab_hip = pd.DataFrame(ringkas)
print('TABEL 6 — Hipotesis per variabel untuk Fase 2')
display(tab_hip)

lb_beda = int((tab_lb[(tab_lb.Deret == 'beda-1')]['Berkorelasi?'] == 'ya').sum())
total_tolak = int((tab_vr['Jalan acak?'] == 'ditolak').sum())
print(f'\nHipotesis jalan acak ditolak pada {total_tolak}/{len(tab_vr)} kombinasi.')
print(f'Deret beda-1 masih berkorelasi pada {lb_beda}/{tab_lb[tab_lb.Deret=="beda-1"].shape[0]} pengujian.')
print()
if total_tolak/len(tab_vr) > 0.3 or lb_beda > 0:
    print('\u2192 LANJUT ke Fase 2. Terdapat struktur yang dapat dipelajari.')
else:
    print('\u2192 BINGKAI ULANG. Deret mendekati jalan acak murni di semua resolusi;')
    print('  paper sebaiknya dilaporkan sebagai temuan negatif yang tetap bernilai.')

### Catatan penting untuk Fase 2

Nilai $VR > 1$ pada deret beriklim harian sebagian besar bersumber dari **siklus
diurnal** — bukan dari pola halus yang hanya dapat ditangkap jaringan saraf.

Karena itu Fase 2 **wajib menyertakan baseline musiman** (nilai pada jam yang
sama kemarin), bukan hanya Naive biasa. Tanpa baseline itu, reviewer akan
menyimpulkan bahwa model hanya mempelajari siklus harian yang sebenarnya dapat
ditangkap metode sepele.

In [ ]:
import shutil
for nm, t in (('tabel1_deskriptif', tab_desk), ('tabel2_kelengkapan', tab_leng),
              ('tabel3_stasioneritas', tab_stat), ('tabel4_ljungbox', tab_lb),
              ('tabel5_variance_ratio', tab_vr), ('tabel6_hipotesis', tab_hip)):
    t.to_csv(f'{OUT}/{nm}.csv', index=False)

shutil.make_archive('hasil_fase1', 'zip', OUT)
print('Berkas keluaran:')
for f in sorted(os.listdir(OUT)):
    print('  ', f)

try:
    from google.colab import files
    files.download('hasil_fase1.zip')
except Exception:
    print('\n(Unduhan otomatis hanya tersedia di Colab.)')